# Quantum-Hybrid Benders Decomposition for Railway Rescheduling — Walkthrough

This notebook is a narrative walkthrough of the pipeline implemented in the `qcedule` package. It covers

1. The Train Rescheduling Problem (TRP) and why it is hard.
2. Classical Benders decomposition into a continuous timing sub-problem and a combinatorial master.
3. The quantum-hybrid variant: where and why quantum helps.
4. The intersection-aware QAOA-style ansatz used for the master.
5. Running the master on IBM Quantum hardware via Qiskit Runtime.
6. Reproducing the headline benchmark plots.
7. Limitations and future work.

All code cells assume the package is installed (`uv sync` or `pip install -e .`) and that the working directory is the repository root. The hardware path requires a `.env` with `QUANTUM_IBM_TOKEN`; everything else runs on a noiseless simulator.

## 1. The Train Rescheduling Problem

After a disturbance — a delayed train, a temporary speed restriction, a closed track segment — the published timetable becomes infeasible. Operators must produce a *new* timetable that

- is **safe** (head-on conflicts and overtakings respect minimum separation),
- **deviates as little as possible** from the original arrivals/departures, and
- can be computed within the disruption response window (seconds, not hours).

Formally we have, for each train $i$ and station $s$ along its route, a real-valued arrival variable $t_{i,s}$. Constraints come in three flavours:

- **Fixed precedences** $t_{j} - t_{i} \ge \tau_{ij}$ — a single train cannot move faster than the line speed.
- **Selectable precedences** — a disjunction of such inequality blocks, modelling a *choice*: which path through a routing area, which train enters a single-track segment first.
- **Deviation budgets** — soft bounds $t_{i,s}$ should not exceed the published time by more than some allowance.

The combinatorial blow-up is in the selectable precedences: each pair of trains conflicting on a shared segment doubles the number of orderings, and each routing area multiplies the number of paths. A direct MILP encoding (`qcedule.centralized.CentralSolver`) scales poorly.

In [ ]:
from qcedule.io_utils.file_parser import parse_network_file, parse_train_file
from qcedule.io_utils.translator import build_constraints

trains = parse_train_file('data/train_OEOEB.txt')
G      = parse_network_file('data/network_OEOEB.txt')
fixed, selectables, deviations = build_constraints(G, trains)

print(f'#trains:       {len(trains)}')
print(f'#fixed prec.:  {len(fixed)}')
print(f'#decisions:    {len(selectables)}  (each is a disjunction of route/order choices)')
print(f'#deviations:   {len(deviations)}')

The Siemens instance ships in `data/network_OEOEB.txt` (the topology) and `data/train_OEOEB.txt` (the published timetable). `build_constraints` walks every train's possible paths through every routing area, derives the set of necessary precedence relations, and adds head-on / overtaking conflicts between every pair of trains.

## 2. Classical Benders decomposition

[Benders, 1962] partitions a problem into a *master* over the hard combinatorial variables and a *sub-problem* over the easy continuous ones. We invert the usual roles: the **continuous** part (timing) becomes the sub-problem solved by an SMT solver, and the **combinatorial** part (which deviation predicates to relax) becomes the master, formulated as Set Cover.

### Sub-problem — Z3 satisfiability

Implemented in `qcedule/satsolving/z3wrapper.py`. The sub-problem encodes every fixed and selectable precedence as a Z3 constraint and every deviation as a *bucketed* predicate $p_{(i,s,\ell)}$ at level $\ell\in\{0,\dots,k-1\}$ such that $p_{(i,s,\ell)} \Rightarrow t_{i,s} \le \text{published} + \ell\,\Delta$. Z3 is invoked with the active predicates as **assumptions**, so an UNSAT response returns a *minimal unsat-core*: a small set of predicates that cannot all be enforced simultaneously. That core is the Benders feasibility cut.

### Master — Set Cover

Implemented in `qcedule/setcovering/scpwrapper.py`. Each cut becomes a row in a Boolean DataFrame whose columns are the deviation predicates. A column "covers" a row if disabling that predicate would relax the corresponding unsat-core. We then ask: *what is the minimum set of predicates we must disable so that every cut so far becomes inactive?* That is exactly the Set Cover Problem.

The full loop in `qcedule/routing.py` is:

```
while True:
    sat, schedule, core = sub.solve(disabled_from=selection)
    if sat:                       # converged
        return schedule
    master.add(core)              # new Benders cut
    selection = master.solve(...) # which predicates to relax next round
```

In [ ]:
# Run the Benders loop with a purely classical (greedy) master to illustrate the loop on the Siemens instance.
from qcedule.routing import benders_algorithm

risks = {
    1: [[6, 21], [33, 0], [24, 7], [27, 14], [16, 30]],
    2: [[6, 21], [33, 0], [24, 7], [27, 14], [16, 30], [17]],
    3: [[6, 21], [33, 0], [24, 7], [27, 14], [16, 30]],
    4: [[27, 14], [16, 30], [17]],
    5: [[6, 21], [33, 0], [24, 7], [27, 14], [16, 30]],
    6: [[27, 14], [24, 7], [6, 21], [33, 0]],
}
constr = (fixed, selectables, deviations)

res = benders_algorithm(constr, platforms=risks, scp_strat='greedy')
print(f'iterations:    {res.iter_num}')
print(f'total dev (s): {res.total_dev}')
print(f'time (s):      {res.time:.2f}')

## 3. Why a quantum master?

The continuous Z3 sub-problem is fast and well-studied — there is no quantum advantage to be had there. The interesting object is the **master SCP**, which (i) is NP-hard in general, (ii) gets *more cuts every iteration*, and (iii) has structure (column overlap) that a constraint-preserving mixer can exploit.

We therefore plug a **Quantum Alternating Operator Ansatz** master in behind the same `Problem.solve(...)` interface as the greedy and Gurobi solvers. The reason this is more than a curiosity is the mixer: by suppressing transitions that would land outside the SCP feasible set, the optimiser explores a much smaller (and structured) Hilbert subspace, which empirically translates into *fewer* Benders iterations — see Section 6.

## 4. Circuit design

The full ansatz lives in `qcedule/setcovering/qasmsolver.py`. For each Benders iteration the SCP DataFrame `data` has

- one *subset qubit* per column (a deviation predicate that could be disabled),
- one *element ancilla* per row (an unsat-core element that must be covered).

Stage by stage:

1. **Initial state** — `X` on every subset qubit. State $|1\dots 1\rangle$ ↔ "all predicates disabled". This is trivially a cover and a natural starting point for the constraint-preserving mixer.
2. **Cost layer** — `RZ(γ)` on each subset qubit, penalising the Hamming weight (number of selected subsets) — i.e. minimum-cardinality covers are favoured.
3. **Mixer layer** (`mc_bitflip_mixer`). For each subset $c$:
   - Build, on each element ancilla $e \in c$, the Boolean disjunction *"some neighbour of $c$ in the constraint graph that contains $e$ is currently selected"*. Neighbours come from `constraint_graph(data)`: an edge between two columns iff their Boolean vectors share at least one element.
   - Apply `RX(β)` on the qubit of $c$, **multi-controlled** by all element ancillas.
   - Uncompute the ancillas with a mirrored chain.

The flip on $c$ thus only fires when *every element* of $c$ is already covered by some neighbour — exactly the condition under which removing $c$ from the cover keeps it feasible. Symmetrically, adding $c$ is unconditional, because adding to a cover never breaks it. This is the constraint-preserving Hadfield-style mixer specialised to set cover.

Below we reconstruct a small SCP DataFrame and visualise its constraint graph — the structure the mixer is built on top of.

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from qcedule.setcovering.qasmsolver import constraint_graph

superset = set(range(14))
subsets  = {
    'a': {i for i in superset if i % 2},
    'b': {i for i in superset if not i % 2},
    'c': {0, 7},
    'd': {1, 2, 8, 9},
    'e': {3, 4, 5, 6, 10, 11, 12, 13},
}
matrix = np.array([[i in subsets[c] for c in subsets] for i in superset])
data   = pd.DataFrame(matrix, columns=list(subsets))

C = constraint_graph(data)
nx.draw_networkx(C, with_labels=True, node_color='#9bd3f5', node_size=900)
plt.title('SCP constraint graph (intersections between subsets)')
plt.axis('off')
plt.show()
data

## 5. Running on IBM hardware

Hardware execution is wrapped in `qcedule/setcovering/qasmrun.py`. The flow is

1. Build the PennyLane circuit on a `lightning.qubit` device with the right wire count.
2. Decompose to `{RX, RY, RZ, CNOT, Toffoli, X}` (`pennylane.transforms.decompose`).
3. Export to **OpenQASM 2** via `QuantumTape.to_openqasm()`.
4. Re-import in Qiskit (`qasm2.loads`), transpile with the preset pass manager (level 1) targeting an IBM backend (`ibm_torino`), and submit to `SamplerV2`.
5. Group counts by the *non-ancilla* bits and pick the most frequent bitstring as the SCP solution.

Hardware runs target **`ibm_torino`** (Heron, 133 qubits). In our experiments the peak problem+ancilla usage was about **38 qubits at difficulty level 15** — well below the 133-qubit ceiling. The transpiled circuit for one such SCP iteration is checked in as `circ` (OpenQASM) and `circ.png` (rendered).

QAOA+ defaults used in the paper: `depth=2`, `shots=10` and `maxiter=10` for the COBYLA parameter optimiser, then `128` shots (with up to 5 retries on transient runtime errors) for the final hardware sample.

To run a single Benders iteration on hardware:

```python
res = benders_algorithm(
    constr, platforms=risks, scp_strat='quantum',
    depth=2, steps=10, simulate=False,   # set simulate=True for noiseless Aer
)
```

Hardware runs require a populated `.env` (`QUANTUM_IBM_TOKEN`).

## 6. Benchmark results

All benchmarks use the Siemens 6-train / 6-station instance with the difficulty knob `EARLY` (the earliest allowed time) sweeping from `15:32:00` to `15:50:00`. Each `Result` object is pickled in `results/`.

In [ ]:
from qcedule.experiments.data import load_results, add_relative_delay
from qcedule.experiments.exp_framework import plot_metric, plot_qubits

qres0 = load_results('results/5_8_11_14_const_QUANT.pkl')
qres1 = load_results('results/6_9_12_15_const_QUANT.pkl')
qres2 = load_results('results/7_10_13_const_QUANT.pkl')
qres3 = load_results('results/0-1-2-3-4_const_QUANT.pkl')
sres  = load_results('results/up_to_13th_constr_SIM.pkl')
sres2 = load_results('results/simulation.pkl')
cres  = load_results('results/presolve_res.pkl')
bres  = load_results('results/greedy_and_gurobi.pkl')
for r in sres + sres2:
    r.method = 'simulator'
qres = qres0 + qres1 + qres2 + qres3 + sres + sres2 + cres + bres
add_relative_delay(qres)

plot_metric(qres, 'iter_num',  ignore_method=['centralized'], minmax=False)
plot_metric(qres, 'total_dev', ignore_method=['greedy', 'gurobi'], minmax=False)
plot_metric(qres, 'max_qubits', ignore_method=['greedy', 'gurobi', 'centralized'], minmax=True)
plot_qubits(qres0 + qres1)

Reading the figures (mirroring the paper's evaluation):

- **`iter_num`** — QAOA+ (both simulator and hardware) converges in **far fewer** master-loop iterations than the greedy or Gurobi master on the harder difficulty levels. The mechanism is counter-intuitive: limited circuit depth (p=2), shot noise, and the short COBYLA budget bias the QAOA+ output distribution toward selecting *slightly larger relaxation sets* than strictly necessary, so the SMT solver encounters fewer fresh incompatibilities per round.
- **`total_dev`** — the centralised Gurobi MILP provides the benchmark minimum deviation. QAOA+ deviations nearly coincide with the classical-Benders curves at low and medium difficulty, with only marginal excess delay at the highest levels. Every method returns feasible, safety-preserving timetables.
- **`max_qubits`** — peak problem+ancilla usage rises to about **38 qubits at difficulty level 15** — well below the 133-qubit ceiling of `ibm_torino`. Allocating additional hardware runtime would extend experiments to even higher difficulties.
- **`plot_qubits`** — qubit count grows near-linearly across Benders iterations, since each new cut adds a row (element ancilla) and possibly a column (subset qubit) to the SCP DataFrame.

Wall-clock runtimes (not plotted here but recorded in the same `Result` objects) tell the opposite story: greedy is the fastest at every difficulty; the centralised Gurobi MILP is the slowest classical method on medium-to-hard instances; QAOA+ runtimes are substantially higher than all classical baselines because of transpilation, network latency, and job execution overhead. This is the principal current limitation of the approach.

## 7. Limitations and future work

**Runtime is not yet competitive.** Quantum simulations on the noise-free AerSimulator are exponentially slower than the real machine; physical hardware runs suffer from decoherence, read-out bias, and CNOT infidelity, and for denser networks the required circuit depth may exceed the device's coherence window. Even after subtracting IBM queue time, end-to-end QAOA+ runtime is much higher than every classical baseline on every instance.

**Qubit footprint is mixer-dominated.** The current QAOA+ implementation maps one problem qubit per SCP subset plus one ancilla qubit per conflict element, which constrains scalability. Future work could investigate more efficient mixer Hamiltonians that reduce the ancilla overhead.

**Parameter optimisation is intentionally cheap.** We cap COBYLA at ten iterations with ten shots per evaluation. For larger problems, gradient-based or Bayesian methods, or more frugal estimators, may converge faster or require fewer circuit evaluations. Parameter-transfer from smaller instances or from simulator-trained values is an obvious next step.

**Modelling assumptions.** The translator assumes deterministic running times and uses a uniform deviation bucket size $\delta_{\max}/K$; adapting the discretisation grid to actual headway distributions or train priorities could reduce the number of Benders iterations. The Z3 sub-problem scales linearly with event count but currently runs single-threaded — a parallel SMT portfolio or hybrid CP–SMT back-end could yield further speed-ups on larger instances.